# Hybrid A* Demo (Educational)

This notebook shows a simplified Hybrid A* planner using states `(row, col, heading_bin)` and motion primitives.

It is designed for intuition, not production-level vehicle planning.

In [ ]:
%matplotlib inline

import heapq
import math

import matplotlib.pyplot as plt
import numpy as np

## 1. Build occupancy grid

In [ ]:
H, W = 45, 45
grid = np.zeros((H, W), dtype=np.uint8)

# Obstacles
grid[8:37, 14] = 1
grid[8:37, 30] = 1
grid[22, 14:31] = 1

# Open corridors
grid[12, 14] = 0
grid[32, 30] = 0
grid[22, 22] = 0

start_xy = (5.0, 5.0)
goal_xy = (39.0, 39.0)
start_heading = math.radians(0.0)
goal_heading = math.radians(0.0)

## 2. State and motion settings

In [ ]:
N_HEADINGS = 16
STEP_LEN = 1.8
STEER_DELTAS = [-math.radians(25), 0.0, math.radians(25)]

def wrap_angle(theta):
    return (theta + 2.0 * math.pi) % (2.0 * math.pi)

def heading_to_bin(theta):
    t = wrap_angle(theta)
    return int(round((t / (2.0 * math.pi)) * N_HEADINGS)) % N_HEADINGS

def bin_to_heading(k):
    return (2.0 * math.pi) * (k / N_HEADINGS)

def to_grid(xy):
    r = int(round(xy[0]))
    c = int(round(xy[1]))
    return r, c

def in_bounds(r, c):
    return 0 <= r < H and 0 <= c < W

def free_cell(r, c):
    return in_bounds(r, c) and grid[r, c] == 0

## 3. Motion primitive expansion

In [ ]:
def primitive_successor(state, steer_delta):
    r, c, hb = state
    theta = bin_to_heading(hb)

    theta2 = wrap_angle(theta + steer_delta)
    nr = r + STEP_LEN * math.sin(theta2)
    nc = c + STEP_LEN * math.cos(theta2)

    gr, gc = to_grid((nr, nc))
    if not free_cell(gr, gc):
        return None

    hb2 = heading_to_bin(theta2)
    return (float(gr), float(gc), hb2), STEP_LEN

def heuristic(state, goal_state):
    r, c, hb = state
    gr, gc, ghb = goal_state

    pos = math.hypot(gr - r, gc - c)
    dhead = abs(hb - ghb)
    dhead = min(dhead, N_HEADINGS - dhead)
    return pos + 0.3 * dhead

## 4. Hybrid A* search

In [ ]:
def reconstruct(parent, node):
    path = [node]
    while node in parent:
        node = parent[node]
        path.append(node)
    path.reverse()
    return path

def hybrid_astar(start_state, goal_state):
    open_heap = []
    parent = {}
    g = {start_state: 0.0}
    closed = set()

    heapq.heappush(open_heap, (heuristic(start_state, goal_state), 0.0, start_state))
    expansions = 0

    while open_heap:
        _, curr_g, current = heapq.heappop(open_heap)
        if current in closed:
            continue

        closed.add(current)
        expansions += 1

        # Accept reaching the goal cell regardless of final heading bin.
        if (int(round(current[0])) == int(round(goal_state[0])) and
            int(round(current[1])) == int(round(goal_state[1]))):
            return reconstruct(parent, current), g[current], expansions

        for steer in STEER_DELTAS:
            nxt = primitive_successor(current, steer)
            if nxt is None:
                continue

            ns, step_cost = nxt
            ng = curr_g + step_cost
            if ng < g.get(ns, math.inf):
                g[ns] = ng
                parent[ns] = current
                f = ng + heuristic(ns, goal_state)
                heapq.heappush(open_heap, (f, ng, ns))

    return None, math.inf, expansions

## 5. Run planner

In [ ]:
start_state = (float(round(start_xy[0])), float(round(start_xy[1])), heading_to_bin(start_heading))
goal_state = (float(round(goal_xy[0])), float(round(goal_xy[1])), heading_to_bin(goal_heading))

path, cost, expansions = hybrid_astar(start_state, goal_state)

print(f'Path found: {path is not None}')
print(f'Path cost: {cost:.2f}')
print(f'Node expansions: {expansions}')

## 6. Visualize result

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(grid, cmap='Greys', origin='upper')

ax.scatter(start_state[1], start_state[0], c='limegreen', s=100, marker='o', label='start')
ax.scatter(goal_state[1], goal_state[0], c='crimson', s=110, marker='*', label='goal')

if path is not None and len(path) > 1:
    xs = [p[1] for p in path]
    ys = [p[0] for p in path]
    ax.plot(xs, ys, color='deepskyblue', linewidth=2.2, label='hybrid path')

    # Draw sparse heading arrows
    stride = max(1, len(path) // 12)
    for p in path[::stride]:
        theta = bin_to_heading(p[2])
        dr = 0.8 * math.sin(theta)
        dc = 0.8 * math.cos(theta)
        ax.arrow(p[1], p[0], dc, dr, color='navy', head_width=0.4, length_includes_head=True)

ax.set_title('Hybrid A* path with heading-aware states')
ax.set_xticks([])
ax.set_yticks([])
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

## 7. Notes

Real Hybrid A* systems often add reverse gear, smoother curvature handling, and post-optimization.

This notebook focuses on the key concept: searching a heading-aware state space with feasible motion primitives.